# SocialChoice 08 - Kemeny, Ranked Pairs et Dodgson : trois regles computationnellement dures (jumeau C# .NET)

**Navigation** : [Predecessor: [ChoixSocial-07-Committees-Core-CSharp]] | [Index](../README.md)

**Autres notebooks** : [ChoixSocial-03-Voting-Methods-CSharp.ipynb](ChoixSocial-03-Voting-Methods-CSharp.ipynb) (regles polynomiales) | [ChoixSocial-04-Computational-Aggregation-SAT-Z3-CSharp.ipynb](ChoixSocial-04-Computational-Aggregation-SAT-Z3-CSharp.ipynb) (encodage SAT des theoremes d'impossibilite) | [Jumeau Python](ChoixSocial-08-Kemeny-RankedPairs-Dodgson-Python.ipynb)

**Kernel** : .net-csharp

***

## Introduction

Le carnet SC-03 a presente les regles de vote **polynomiales** (pluralite, Borda, Copeland, IRV). Ce carnet-ci presente les **trois regles canoniques computationnellement dures** :

- **Kemeny** (Kemeny 1959 ; Handbook ch. 2 §2.7, ch. 4 §4.1-4.2) -- regle du **rang medianique** : choisir le classement qui maximise le nombre d'accords pairwise avec les preferences des electeurs (Kemeny score). Caracterisation axiomatique propre par Young-Levenglick (1978) : Kemeny est l'unique regle qui satisfait a la fois **Condorcet-coherence** et **neutralite**.
- **Ranked Pairs** (Tideman 1987 ; Handbook ch. 4 §4.5, Theoreme 4.15, Brill et Fischer 2012) -- trier les duels pairwise par marge decroissante, verrouiller les paires dans l'ordre tant qu'on ne cree pas de cycle (algorithme de Tideman). **Nuance de complexite** : construire le classement avec un **departage fixe** est polynomial (tri + verrouillage, ce que le carnet implemente) ; decider si une alternative est un gagnant sur l'**ensemble des departages** est **NP-complet** (Brill et Fischer 2012).
- **Dodgson** (Dodgson 1876 ; Handbook ch. 5, §5.3 Winner-Problem Complexity) -- pour chaque candidat x, le **Dodgson score** est le nombre minimal d'inversions de paires adjacentes dans les bulletins individuels necessaires pour faire de x le vainqueur de Condorcet. Le gagnant est celui de score minimum. Winner-Determination est **NP-dur** (Handbook ch. 5 §5.3 Winner-Problem Complexity, Caragiannis-E.Hemaspaandra-L.Hemaspaandra 2016).

Ces trois regles sont le **contrepoint computationnel** des regles polynomiales de SC-03 : un moteur SOTA (Z3, solveur ILP, ou tout SAT solver) prend tout son sens ici -- la complexite NP-dure est structurellement un probleme non trivial (axe 2 de #3801).

### Prerequis : SC-03 et SC-04

- SC-03 : cycle de Condorcet, duels pairwise, regles de vote classiques.
- SC-04 : encodage SAT/Z3 des theoremes d'impossibilite (Arrow, Sen).

### Duree estimee : 35 minutes

> *Ancres savantes -- Kemeny, J.G. (1959), Mathematics without numbers, Daedalus 88(4):577-591 (Kemeny score : classement qui maximise la somme des accords pairwise, la regle de Kemeny) ; Young, H.P. & Levenglick, A. (1978), A consistent extension of Condorcet's election principle, SIAM Journal on Applied Mathematics 35(2):285-300 (caracterisation axiomatique de Kemeny : unique regle a satisfaire Condorcet-coherence + neutralite, voir aussi Handbook ch. 4 §4.1-4.2) ; Tideman, T.N. (1987), Independence of clones as a criterion for voting rules, Social Choice and Welfare 4(3):185-206 (Ranked Pairs, Handbook ch. 4 §4.5, Theoreme 4.15, Brill et Fischer 2012) ; Dodgson, C. (1876), A method of taking votes upon more than two issues, manuscrit non publie, reimprime dans McLean, I. & Urken, A.B. (1995), Classics of Social Choice, University of Michigan Press, pp. 273-287 (Dodgson score, manuscrit redige par l'auteur d'Alice au pays des merveilles ; voir Handbook ch. 5 §5.3) ; Caragiannis, I., Hemaspaandra, E. & Hemaspaandra, L. (2016), Dodgson's Rule and Young's Rule, in Handbook of Computational Social Choice (Brandt et al. ed.), ch. 5, pp. 121-143, Cambridge University Press, ISBN 978-1-107-06043-2 (NP-durete de la determination du gagnant Dodgson, §5.3 Winner-Problem Complexity) ; Brandt, F., Conitzer, V., Endriss, U., Lang, J., Procaccia, A.D. (2016), Handbook of Computational Social Choice, Cambridge University Press, ISBN 978-1-107-06043-2 (ouvrage d'ancrage, ch. 4 §4.5 ranked pairs, ch. 2 §2.7 Kemeny regle, ch. 4 §4.1-4.2 Kemeny, ch. 5 §5.3 Dodgson et NP-durete, source biblio cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Social Choice, Voting and Welfare\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`).*

***

## 0. Configuration et imports

Ce carnet utilise les tableaux 2D et LINQ pour les manipulations de matrices et l'enumeration recursive des permutations (Kemeny est resolu par enumeration exhaustive des `m!` permutations possibles).

In [1]:
#r "nuget: Microsoft.Z3, 4.12.2"
#load "../../SymbolicAI/SMT/Z3-API/Z3NativeLoader.cs"
// Configuration et imports (chargement Z3 canonique de la serie, miroir SC-04)
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using Microsoft.Z3;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

public static partial class SocialChoice08
{
    /// <summary>
    /// Enumeration recursive de toutes les permutations, dans l'ordre de la liste source.
    /// </summary>
    public static IEnumerable<List<string>> Permutations(IReadOnlyList<string> alternatives)
    {
        if (alternatives.Count == 0)
        {
            yield return new List<string>();
            yield break;
        }

        for (int i = 0; i < alternatives.Count; i++)
        {
            var remaining = new List<string>(alternatives);
            string chosen = remaining[i];
            remaining.RemoveAt(i);
            foreach (var tail in Permutations(remaining))
            {
                var permutation = new List<string> { chosen };
                permutation.AddRange(tail);
                yield return permutation;
            }
        }
    }

    /// <summary>
    /// Calcule la matrice des marges pairwise N(a_i, a_j).
    /// N(a, b) = (nb votants preferant a a b) - (nb votants preferant b a a).
    /// Convention : N(a, b) > 0 signifie majorite pour a.
    /// </summary>
    public static int[,] PairwiseMargin(
        IReadOnlyList<List<string>> profile,
        IReadOnlyList<string> alternatives)
    {
        int m = alternatives.Count;
        var margins = new int[m, m];

        for (int i = 0; i < m; i++)
        {
            for (int j = 0; j < m; j++)
            {
                if (i == j) continue;

                string a = alternatives[i];
                string b = alternatives[j];
                int aWins = profile.Count(pref => pref.IndexOf(a) < pref.IndexOf(b));
                int bWins = profile.Count(pref => pref.IndexOf(b) < pref.IndexOf(a));
                margins[i, j] = aWins - bWins;
            }
        }

        return margins;
    }
}

Console.WriteLine("Configuration OK : SocialChoice 08 - Kemeny, Ranked Pairs, Dodgson (jumeau C# .NET)");
Console.WriteLine($"Runtime .NET version : {Environment.Version}");
Console.WriteLine($"Microsoft.Z3 version : {typeof(Context).Assembly.GetName().Version}");
Console.WriteLine($"Z3 version : {Microsoft.Z3.Version.FullVersion}");
Console.WriteLine($"Culture : {CultureInfo.CurrentCulture.Name}");

Installed Packages Microsoft.Z3, 4.12.2

Configuration OK : SocialChoice 08 - Kemeny, Ranked Pairs, Dodgson (jumeau C# .NET)


Runtime .NET version : 10.0.13


Microsoft.Z3 version : 4.12.2.0


Z3 version : Z3 4.12.2.0


Culture : 


### Configuration de l'environnement

Le carnet utilise principalement les tableaux 2D pour les calculs de matrices pairwise et l'enumeration recursive des permutations pour enumerer les classements. Pour `m <= 6` alternatives, l'enumeration directe reste rapide ; au-dela, il faut basculer sur une formulation ILP/SAT, presentee en section 4.

***

## 1. Regle de Kemeny : Kemeny score exact par enumeration des permutations

### 1.1 Definition : Kemeny score

Pour un profil de preferences `P` sur un ensemble d'alternatives `A`, le **Kemeny score** d'un classement `sigma` (une permutation de `A`) est :

```
kemeny_score(sigma, P) = somme_{x avant y dans sigma} nombre de votants preferant x a y
```

Ce score compte les accords individuels, y compris lorsque les duels sont a egalite.

Le **Kemeny winner** est l'ensemble des alternatives `a` qui peuvent figurer en tete d'un classement de score maximum.

In [2]:
// Nombre total d'accords entre le classement et les bulletins.
public static class KemenyEngine
{
    public static int KemenyScoreOfRanking(
        IReadOnlyList<string> ranking,
        IReadOnlyList<List<string>> profile)
    {
        int score = 0;
        for (int i = 0; i < ranking.Count; i++)
        {
            for (int j = i + 1; j < ranking.Count; j++)
            {
                string x = ranking[i];
                string y = ranking[j];
                score += profile.Count(pref => pref.IndexOf(x) < pref.IndexOf(y));
            }
        }
        return score;
    }

    // Kemeny winner exact par enumeration exhaustive des permutations.
    // Pour |A| = m, il y a m! classements possibles. On prend ceux de
    // score Kemeny maximum et on retient tous les candidats qui peuvent
    // figurer en tete d'un tel classement optimal.
    public static (int bestScore, List<List<string>> optimalRankings, List<string> winners)
    KemenyBruteForce(
        IReadOnlyList<List<string>> profile,
        IReadOnlyList<string> alternatives)
    {
        int m = alternatives.Count;
        if (m > 8)
        {
            throw new ArgumentOutOfRangeException(
                nameof(alternatives),
                $"Enumeration prohibitive pour m={m} (>8 alternatives)");
        }

        int bestScore = int.MinValue;
        var optimalRankings = new List<List<string>>();

        foreach (var ranking in SocialChoice08.Permutations(alternatives))
        {
            int score = KemenyScoreOfRanking(ranking, profile);
            if (score > bestScore)
            {
                bestScore = score;
                optimalRankings.Clear();
                optimalRankings.Add(ranking);
            }
            else if (score == bestScore)
            {
                optimalRankings.Add(ranking);
            }
        }

        var winners = optimalRankings
            .Select(ranking => ranking[0])
            .Distinct()
            .OrderBy(candidate => candidate, StringComparer.Ordinal)
            .ToList();

        return (bestScore, optimalRankings, winners);
    }
}

// Test 1 : profil classique de Condorcet (cycle symetrique)
var condorcetProfile = new List<List<string>>
{
    new() { "A", "B", "C" },
    new() { "B", "C", "A" },
    new() { "C", "A", "B" },
};
var alts = new List<string> { "A", "B", "C" };

var (score, optimal, winners) = KemenyEngine.KemenyBruteForce(condorcetProfile, alts);
Console.WriteLine("PROFIL DE CONDORCET (cycle symetrique A > B > C > A)");
Console.WriteLine(new string('=', 60));
for (int i = 0; i < condorcetProfile.Count; i++)
    Console.WriteLine($"  Votant {i + 1}: {string.Join(" > ", condorcetProfile[i])}");
Console.WriteLine();
Console.WriteLine($"Kemeny score max : {score}");
Console.WriteLine($"Classements optimaux ({optimal.Count}) :");
foreach (var ranking in optimal)
    Console.WriteLine($"  {string.Join(" > ", ranking)}");
Console.WriteLine();
Console.WriteLine($"Kemeny winners : [{string.Join(", ", winners)}]");

PROFIL DE CONDORCET (cycle symetrique A > B > C > A)


  Votant 1: A > B > C


  Votant 2: B > C > A


  Votant 3: C > A > B


Kemeny score max : 5


Classements optimaux (3) :


  A > B > C


  B > C > A


  C > A > B


Kemeny winners : [A, B, C]


### Interpretation : Kemeny sur le cycle de Condorcet

Les trois classements optimaux suivent chacun deux arcs du cycle. Leurs premiers candidats sont distincts : les trois candidats sont donc vainqueurs Kemeny. Le score compte les accords individuels sur toutes les paires.

L’énumeration examine `m!` classements. La formulation par optimisation evite de les enumerer explicitement, mais la determination d’un vainqueur Kemeny reste un probleme NP-dur.

In [3]:
// Test 2 : profil divergent de SC-03 (Borda et Copeland donnent B)
var divergentProfile = new List<List<string>>
{
    new() { "A", "B", "C" },
    new() { "A", "B", "C" },
    new() { "B", "C", "A" },
    new() { "C", "B", "A" },
    new() { "C", "B", "A" },
};

var (scoreD, optimalD, winnersD) = KemenyEngine.KemenyBruteForce(divergentProfile, alts);
Console.WriteLine("PROFIL DIVERGENT (SC-03 : Borda + Copeland = B)");
Console.WriteLine(new string('=', 60));
for (int i = 0; i < divergentProfile.Count; i++)
    Console.WriteLine($"  Votant {i + 1}: {string.Join(" > ", divergentProfile[i])}");
Console.WriteLine();
Console.WriteLine($"Kemeny score max : {scoreD}");
Console.WriteLine($"Classements optimaux ({optimalD.Count}) :");
foreach (var ranking in optimalD)
    Console.WriteLine($"  {string.Join(" > ", ranking)}");
Console.WriteLine();
Console.WriteLine($"Kemeny winners : [{string.Join(", ", winnersD)}]");

PROFIL DIVERGENT (SC-03 : Borda + Copeland = B)


  Votant 1: A > B > C


  Votant 2: A > B > C


  Votant 3: B > C > A


  Votant 4: C > B > A


  Votant 5: C > B > A


Kemeny score max : 9


Classements optimaux (1) :


  B > C > A


Kemeny winners : [B]


### Interpretation : Kemeny sur le profil divergent

Le classement optimal est B > C > A : B bat chacun de ses concurrents et C bat A. B est donc l'unique vainqueur Kemeny ; Borda et Copeland donnent aussi B sur ce profil.


***

## 2. Ranked Pairs (Tideman 1987)

### 2.1 Algorithme

1. Pour chaque paire `(x, y)`, calculer la marge `N(x, y)` (positive : x l'emporte sur y par majorite).
2. Trier les paires par marge **decroissante** (tie-break par ordre alphabetique pour reproductibilite).
3. Pour chaque paire dans l'ordre, **verrouiller** la paire dans le graphe oriente SAUF si cela cree un cycle.

Le graphe final est acyclique ; un tri topologique donne un classement compatible. Les vainqueurs sont ses sources (candidats sans arc entrant), qui peuvent etre multiples si des duels sont a egalite.


In [4]:
// === Ranked Pairs (Tideman 1987) -- port C# du twin Python ===
public static class RankedPairsEngine
{
    /// <summary>
    /// Verifie si target peut atteindre start dans le graphe.
    /// Si oui, ajouter start -> target creerait un cycle.
    /// </summary>
    public static bool CreatesCycle(
        Dictionary<string, HashSet<string>> graph,
        string start,
        string target)
    {
        var visited = new HashSet<string>();
        var stack = new Stack<string>();
        stack.Push(target);

        while (stack.Count > 0)
        {
            string node = stack.Pop();
            if (node == start) return true;
            if (!visited.Add(node)) continue;
            if (graph.TryGetValue(node, out var neighbors))
            {
                foreach (string neighbor in neighbors)
                    if (!visited.Contains(neighbor)) stack.Push(neighbor);
            }
        }

        return false;
    }

    /// <summary>
    /// Regle des Ranked Pairs (Tideman 1987).
    /// - ordering : extension topologique du graphe verrouille (liste)
    /// - winners : ensemble des sources (candidats sans arc entrant)
    /// </summary>
    public static (List<string> ordering, List<string> winners) RankedPairs(
        IReadOnlyList<List<string>> profile,
        IReadOnlyList<string> alternatives)
    {
        int m = alternatives.Count;
        int[,] margins = SocialChoice08.PairwiseMargin(profile, alternatives);

        var pairsSorted = new List<(int margin, string x, string y)>();
        for (int i = 0; i < m; i++)
        {
            for (int j = 0; j < m; j++)
            {
                if (i == j) continue;
                int margin = margins[i, j];
                if (margin > 0) pairsSorted.Add((margin, alternatives[i], alternatives[j]));
            }
        }
        pairsSorted = pairsSorted
            .OrderByDescending(p => p.margin)
            .ThenBy(p => p.x, StringComparer.Ordinal)
            .ThenBy(p => p.y, StringComparer.Ordinal)
            .ToList();

        var graph = new Dictionary<string, HashSet<string>>();
        var inDegree = alternatives.ToDictionary(a => a, a => 0);

        foreach (var pair in pairsSorted)
        {
            if (CreatesCycle(graph, pair.x, pair.y)) continue;
            if (!graph.TryGetValue(pair.x, out var targets))
            {
                targets = new HashSet<string>();
                graph[pair.x] = targets;
            }
            targets.Add(pair.y);
            inDegree[pair.y]++;
        }

        var visited = new HashSet<string>();
        var order = new List<string>();

        void Dfs(string node)
        {
            if (visited.Contains(node)) return;
            visited.Add(node);
            if (graph.TryGetValue(node, out var neighbors))
            {
                foreach (string neighbor in neighbors.OrderBy(n => n, StringComparer.Ordinal))
                    Dfs(neighbor);
            }
            order.Add(node);
        }

        foreach (string a in alternatives.OrderBy(a => a, StringComparer.Ordinal))
            Dfs(a);

        order.Reverse();

        var winners = alternatives
            .Where(a => inDegree[a] == 0)
            .OrderBy(a => a, StringComparer.Ordinal)
            .ToList();

        return (order, winners);
    }
}

// Test sur le profil divergent
var (orderRp, winnersRp) = RankedPairsEngine.RankedPairs(divergentProfile, alts);
Console.WriteLine("PROFIL DIVERGENT : Ranked Pairs");
Console.WriteLine(new string('=', 60));
for (int i = 0; i < divergentProfile.Count; i++)
    Console.WriteLine($"  Votant {i + 1}: {string.Join(" > ", divergentProfile[i])}");
Console.WriteLine();
Console.WriteLine($"Ranked Pairs order : {string.Join(" > ", orderRp)}");
Console.WriteLine($"Ranked Pairs winners : [{string.Join(", ", winnersRp)}]");

// Test sur le profil cyclique de Condorcet
var (orderRpC, winnersRpC) = RankedPairsEngine.RankedPairs(condorcetProfile, alts);
Console.WriteLine();
Console.WriteLine("PROFIL DE CONDORCET : Ranked Pairs");
Console.WriteLine(new string('=', 60));
for (int i = 0; i < condorcetProfile.Count; i++)
    Console.WriteLine($"  Votant {i + 1}: {string.Join(" > ", condorcetProfile[i])}");
Console.WriteLine();
Console.WriteLine($"Ranked Pairs order : {string.Join(" > ", orderRpC)}");
Console.WriteLine($"Ranked Pairs winners : [{string.Join(", ", winnersRpC)}]");

PROFIL DIVERGENT : Ranked Pairs


  Votant 1: A > B > C


  Votant 2: A > B > C


  Votant 3: B > C > A


  Votant 4: C > B > A


  Votant 5: C > B > A


... (4 outputs left out)

  Votant 2: B > C > A


  Votant 3: C > A > B


Ranked Pairs order : A > B > C


Ranked Pairs winners : [A]


### Interpretation : Ranked Pairs sur les deux profils

Sur le profil divergent, les trois majorites sont verrouillees et B est la seule source. Sur le cycle, le departage alphabetique verrouille A > B puis B > C ; C > A est refuse car cet arc fermerait un cycle. A est alors la seule source. Un autre departage des marges egales peut choisir une autre source.

Ranked Pairs respecte le vainqueur de Condorcet : sur le profil divergent, B bat A et C et reste l'unique source.


***

## 3. Regle de Dodgson (1876)

### 3.1 Definition : Dodgson score

Pour un profil `P` et un candidat `x`, le **Dodgson score** `DS(x, P)` est le nombre minimal d'inversions de paires adjacentes dans les preferences individuelles (cumule sur tous les electeurs) necessaire pour faire de `x` le vainqueur de Condorcet sur le profil modifie.

Le **Dodgson winner** est l'ensemble des candidats `x` de score minimum.

### 3.2 Implementation directe

Pour chaque candidat et chaque bulletin, il suffit de considerer sa remontee par swaps adjacents : les autres swaps ne peuvent ameliorer ses duels. Une programmation dynamique conserve le cout minimal pour chaque vecteur de victoires, chaque coordonnee etant plafonnee au seuil de majorite stricte. Son espace croît exponentiellement avec le nombre de concurrents : cette methode convient aux petits profils.


In [5]:
// === Regle de Dodgson (1876) -- port C# du twin Python ===
public static class DodgsonEngine
{
    /// <summary>
    /// Verifie si candidate est Condorcet winner dans profile.
    /// </summary>
    public static bool IsCondorcetWinner(
        string candidate,
        IReadOnlyList<List<string>> profile,
        IReadOnlyList<string> alternatives)
    {
        foreach (string other in alternatives)
        {
            if (other == candidate) continue;
            int cWins = profile.Count(p => p.IndexOf(candidate) < p.IndexOf(other));
            if (cWins <= profile.Count / 2.0) return false;
        }
        return true;
    }

    /// <summary>
    /// Nombre minimal de swaps adjacents pour rendre candidate Condorcet.
    /// </summary>
    public static int DodgsonScore(
        string candidate,
        IReadOnlyList<List<string>> profile,
        IReadOnlyList<string> alternatives)
    {
        var opponents = alternatives.Where(a => a != candidate).ToList();
        int threshold = profile.Count / 2 + 1;
        var states = new Dictionary<string, int>
        {
            { string.Join(",", opponents.Select(_ => 0)), 0 }
        };

        foreach (var pref in profile)
        {
            int position = pref.IndexOf(candidate);
            var nextStates = new Dictionary<string, int>();

            foreach (var entry in states)
            {
                int[] counts = entry.Key.Length == 0
                    ? Array.Empty<int>()
                    : entry.Key.Split(',').Select(int.Parse).ToArray();

                for (int shift = 0; shift <= position; shift++)
                {
                    int newPosition = position - shift;
                    int[] wins = opponents
                        .Select(a => pref.IndexOf(a) >= newPosition ? 1 : 0)
                        .ToArray();

                    var updated = new int[counts.Length];
                    for (int k = 0; k < counts.Length; k++)
                        updated[k] = Math.Min(threshold, counts[k] + wins[k]);

                    string key = string.Join(",", updated);
                    int newCost = entry.Value + shift;
                    if (!nextStates.TryGetValue(key, out int best) || newCost < best)
                        nextStates[key] = newCost;
                }
            }

            states = nextStates;
        }

        string targetKey = string.Join(",", opponents.Select(_ => threshold));
        return states.TryGetValue(targetKey, out int cost) ? cost : -1;
    }

    /// <summary>
    /// Dodgson winner exact par programmation dynamique.
    /// </summary>
    public static (int minScore, Dictionary<string, int> scores, List<string> winners) BruteForce(
        IReadOnlyList<List<string>> profile,
        IReadOnlyList<string> alternatives)
    {
        var scores = new Dictionary<string, int>();
        foreach (string candidate in alternatives)
            scores[candidate] = DodgsonScore(candidate, profile, alternatives);

        var validScores = scores.Values.Where(s => s >= 0).ToList();
        if (validScores.Count == 0) return (-1, scores, new List<string>());

        int minScore = validScores.Min();
        var winners = scores
            .Where(kv => kv.Value == minScore)
            .Select(kv => kv.Key)
            .OrderBy(a => a, StringComparer.Ordinal)
            .ToList();

        return (minScore, scores, winners);
    }
}

// Test 1 : profil divergent
Console.WriteLine("DODGSON SUR LE PROFIL DIVERGENT");
Console.WriteLine(new string('=', 60));
for (int i = 0; i < divergentProfile.Count; i++)
    Console.WriteLine($"  Votant {i + 1}: {string.Join(" > ", divergentProfile[i])}");
Console.WriteLine();
var (dsScore, dsAll, dsWinners) = DodgsonEngine.BruteForce(divergentProfile, alts);
foreach (var kv in dsAll)
    Console.WriteLine($"  Dodgson score({kv.Key}) = {kv.Value}");
Console.WriteLine();
Console.WriteLine($"Dodgson winners : [{string.Join(", ", dsWinners)}]");

// Test 2 : profil de Condorcet
Console.WriteLine();
Console.WriteLine("DODGSON SUR LE PROFIL DE CONDORCET");
Console.WriteLine(new string('=', 60));
var (dsC, dsAllC, dsWinnersC) = DodgsonEngine.BruteForce(condorcetProfile, alts);
foreach (var kv in dsAllC)
    Console.WriteLine($"  Dodgson score({kv.Key}) = {kv.Value}");
Console.WriteLine();
Console.WriteLine($"Dodgson winners : [{string.Join(", ", dsWinnersC)}]");

DODGSON SUR LE PROFIL DIVERGENT


  Votant 1: A > B > C


  Votant 2: A > B > C


  Votant 3: B > C > A


  Votant 4: C > B > A


  Votant 5: C > B > A


... (5 outputs left out)

  Dodgson score(A) = 1


  Dodgson score(B) = 1


  Dodgson score(C) = 1


Dodgson winners : [A, B, C]


### Interpretation : Dodgson winner sur les deux profils

**Profil divergent** : B est le Dodgson winner (score 0, deja Condorcet winner). C a besoin d'un swap et A de deux swaps pour devenir Condorcet winner. C'est coherent avec toutes les autres regles (Borda, Copeland, Kemeny, RP) : **B** est l'unique gagnant.

**Profil de Condorcet** : aucun candidat n'est Condorcet winner au depart. Le Dodgson score est le nombre minimal de swaps pour chacun :
- A : dans B > C > A, echanger C et A donne B > A > C.
- B : dans A > B > C, echanger A et B donne B > A > C.
- C : dans A > B > C, echanger B et C donne A > C > B.

Dans chaque cas, le candidat modifie gagne alors ses deux duels.

Les trois candidats ont le meme Dodgson score 1, donc tous sont Dodgson winners (egalite etalee, le cycle est symetrique). C'est coherent avec l'indecision attendue sur un cycle pur.

**Note sur la complexite** : les etats memorisent les victoires contre chaque concurrent, plafonnees au seuil de majorite. Leur nombre peut croitre exponentiellement avec le nombre d'alternatives ; la methode est destinee aux petites instances.


***

## 4. Confrontation Z3 du Kemeny winner (esprit de SC-04)

Pour mettre en valeur un solveur, il faut un probleme **non trivial**. Nous encodons le Kemeny winner comme un probleme d'optimisation Z3 : trouver la permutation sigma qui maximise le Kemeny score. Enumeration classique = `m!` permutations ; formulation Z3 = une optimisation globale puis une optimisation par candidat.


In [6]:
// Encodage Z3 du Kemeny winner
// Microsoft.Z3 4.12.2 (NuGet) est charge par la cellule de configuration.
Console.WriteLine("z3 disponible pour encodage Kemeny winner");
Console.WriteLine($"Assembly Microsoft.Z3 : {typeof(Microsoft.Z3.Context).Assembly.GetName().Name} {typeof(Microsoft.Z3.Context).Assembly.GetName().Version}");
Console.WriteLine($"API d'optimisation : {typeof(Microsoft.Z3.Optimize).FullName}");

z3 disponible pour encodage Kemeny winner


Assembly Microsoft.Z3 : Microsoft.Z3 4.12.2.0


API d'optimisation : Microsoft.Z3.Optimize


### Implementation Z3 : Kemeny winner comme optimisation

On utilise `Microsoft.Z3.Optimize` (solveur d'optimisation SMT) pour trouver le Kemeny score maximum sur des alternatives permutees. Le modele z3 donne la permutation gagnante. Pour chaque candidat, une optimisation supplementaire verifie s'il peut mener un classement optimal.


In [7]:
#nullable enable

using Microsoft.Z3;

// === Kemeny winner via Z3 (Microsoft.Z3 Optimize) -- port C# du twin Python ===
public static class KemenyZ3Solver
{
    /// <summary>
    /// Kemeny winner exact via Z3 Optimize.
    /// Encode chaque permutation sigma comme des variables entieres de rang,
    /// et maximise le Kemeny score.
    /// </summary>
    public static (string status, int score, List<string> ranking, List<string> winners) Solve(
        IReadOnlyList<List<string>> profile,
        IReadOnlyList<string> alternatives)
    {
        int m = alternatives.Count;
        if (m > 6)
            return ("TIMEOUT", -1, new List<string>(), new List<string>());

        using var ctx = new Context();
        var rank = alternatives.ToDictionary(a => a, a => ctx.MkIntConst($"r_{a}"));

        var constraints = new List<BoolExpr>();
        foreach (string a in alternatives)
        {
            constraints.Add(ctx.MkGe(rank[a], ctx.MkInt(0)));
            constraints.Add(ctx.MkLt(rank[a], ctx.MkInt(m)));
        }
        for (int i = 0; i < m; i++)
        {
            for (int j = i + 1; j < m; j++)
                constraints.Add(ctx.MkNot(ctx.MkEq(rank[alternatives[i]], rank[alternatives[j]])));
        }

        var terms = new List<ArithExpr>();
        for (int i = 0; i < m; i++)
        {
            for (int j = i + 1; j < m; j++)
            {
                string x = alternatives[i];
                string y = alternatives[j];
                int cxy = profile.Count(p => p.IndexOf(x) < p.IndexOf(y));
                int cyx = profile.Count(p => p.IndexOf(y) < p.IndexOf(x));
                terms.Add((ArithExpr)ctx.MkITE(ctx.MkLt(rank[x], rank[y]), ctx.MkInt(cxy), ctx.MkInt(cyx)));
            }
        }
        ArithExpr score = ctx.MkAdd(terms.ToArray());

        (int score, List<string> ranking)? SolveWith(string? first)
        {
            var opt = ctx.MkOptimize();
            foreach (BoolExpr constraint in constraints) opt.Add(constraint);
            if (first != null) opt.Add(ctx.MkEq(rank[first], ctx.MkInt(0)));
            opt.MkMaximize(score);

            if (opt.Check() != Status.SATISFIABLE) return null;
            Model model = opt.Model;
            int value = (int)((IntNum)model.Eval(score, true)).Int64;
            var ranking = alternatives
                .OrderBy(a => ((IntNum)model.Eval(rank[a], true)).Int64)
                .ToList();
            return (value, ranking);
        }

        var global = SolveWith(null);
        if (global == null) return ("UNKNOWN", -1, new List<string>(), new List<string>());

        int optimalScore = global.Value.score;
        List<string> optimalRanking = global.Value.ranking;
        var winners = new List<string>();

        foreach (string alt in alternatives)
        {
            var constrained = SolveWith(alt);
            if (constrained == null) return ("UNKNOWN", -1, new List<string>(), new List<string>());
            if (constrained.Value.score == optimalScore) winners.Add(alt);
        }

        return ("OPTIMAL", optimalScore, optimalRanking, winners);
    }
}

// Test 1 : profil divergent
Console.WriteLine("KEMENY VIA Z3 (profil divergent)");
Console.WriteLine(new string('=', 60));
var resultZ3 = KemenyZ3Solver.Solve(divergentProfile, alts);
Console.WriteLine($"Status : {resultZ3.status}");
if (resultZ3.status == "OPTIMAL")
{
    Console.WriteLine($"Score optimal : {resultZ3.score}");
    Console.WriteLine($"Classement optimal : {string.Join(" > ", resultZ3.ranking)}");
    Console.WriteLine($"Kemeny winners : [{string.Join(", ", resultZ3.winners)}]");
}
Console.WriteLine($"Controle enumeration vs Z3 : enumeration = {scoreD}, Z3 = {resultZ3.score}, concordant = {scoreD == resultZ3.score}");

// Test 2 : profil de Condorcet
Console.WriteLine();
Console.WriteLine("KEMENY VIA Z3 (profil de Condorcet)");
Console.WriteLine(new string('=', 60));
var resultZ3C = KemenyZ3Solver.Solve(condorcetProfile, alts);
Console.WriteLine($"Status : {resultZ3C.status}");
if (resultZ3C.status == "OPTIMAL")
{
    Console.WriteLine($"Score optimal : {resultZ3C.score}");
    Console.WriteLine($"Classement optimal : {string.Join(" > ", resultZ3C.ranking)}");
    Console.WriteLine($"Kemeny winners : [{string.Join(", ", resultZ3C.winners)}]");
}
Console.WriteLine($"Controle enumeration vs Z3 : enumeration = {score}, Z3 = {resultZ3C.score}, concordant = {score == resultZ3C.score}");

KEMENY VIA Z3 (profil divergent)


Status : OPTIMAL


Score optimal : 9


Classement optimal : B > C > A


Kemeny winners : [B]


Controle enumeration vs Z3 : enumeration = 9, Z3 = 9, concordant = True


... (2 outputs left out)

Score optimal : 5


Classement optimal : C > A > B


Kemeny winners : [A, B, C]


Controle enumeration vs Z3 : enumeration = 5, Z3 = 5, concordant = True


### Interpretation : Kemeny via Z3 sur les deux profils

L'enumeration et Z3 trouvent le meme score optimal et le meme ensemble de vainqueurs pour chacun des deux profils. Sur le profil divergent, B est seul vainqueur. Sur le cycle, A, B et C peuvent chacun mener un classement optimal. Le classement affiche par Z3 est un representant optimal ; il peut varier selon le solveur.

Pour verifier chaque candidat, on maximise a nouveau le score en imposant sa premiere place. Il est vainqueur exactement lorsque cet optimum contraint egale l'optimum global.


***

## 5. Resume

Ce carnet a presente les trois regles computationnellement dures du choix social (Kemeny NP-dur, Ranked Pairs polynomial avec departage fixe mais NP-complet sur l'ensemble des departages, Dodgson NP-dur) et les a confrontees a une formulation Z3. Les implementations C# sont executees (Kemeny par enumeration des permutations, Ranked Pairs par algorithme de Tideman, Dodgson par programmation dynamique sur les victoires pairwise). La formulation Z3 du Kemeny winner utilise `Microsoft.Z3.Optimize` et confirme le resultat brut-force sur les deux profils tests.

| Regle | Complexite winner determination | Source |
|---|---|---|
| Kemeny | NP-dur (exact par enumeration : `m!` permutations) | Kemeny 1959, Handbook ch. 4 §4.1-4.2 |
| Ranked Pairs | **Polynomial** (tri + verrouillage, departage fixe) ; **NP-complet** pour decider si une alternative est gagnante sur l'ensemble des departages | Tideman 1987, Handbook ch. 4 §4.5 (Thm 4.15, Brill-Fischer 2012) |
| Dodgson | NP-dur (programmation dynamique sur les victoires pour les petits profils) | Dodgson 1876, Handbook ch. 5 §5.3, Caragiannis-E.Hemaspaandra-L.Hemaspaandra 2016 |

**Ancrage bibliographique** : le Handbook of Computational Social Choice (Brandt-Conitzer-Endriss-Lang-Procaccia, 2016, ISBN 978-1-107-06043-2) est la reference canonique pour ces trois regles. Source biblio cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Social Choice, Voting and Welfare\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`. Les trois regles presentent un **probleme non trivial** (axe 2 de #3801) : la determination du gagnant est structurellement NP-dure (Kemeny, Dodgson), ce qui fait du solveur un outil preuve du contenu - et non un gadget.


***

## Exercices

**Note pedagogique** : les exercices suivent la convention `// TODO etudiant` + message console + valeur neutre (`return null`, liste vide) (regle C.1), pas de `NotImplementedException`.


### Exemple guide : generation d'un profil aleatoire

**Contexte** : `random_profile` tire, pour `n` electeurs, un classement aleatoire de toutes les `m` alternatives (sans remplacement, via un melange de Fisher-Yates sur `System.Random`). Le profil ainsi produit est la matiere premiere de l'exercice 1 ci-dessous : chaque electeur est une liste de classements, de la preference la plus forte a la plus faible.


In [8]:
// Exemple guide : generation d'un profil aleatoire
// RNG .NET : System.Random(seed) est deterministe pour une graine fixee,
// la graine etant propre au carnet C#.
static List<List<string>> RandomProfile(int nVoters, List<string> alternatives, int? seed = null)
{
    var rng = seed.HasValue ? new Random(seed.Value) : new Random();
    var profile = new List<List<string>>();

    for (int voter = 0; voter < nVoters; voter++)
    {
        var ranking = new List<string>(alternatives);
        // Melange de Fisher-Yates : permutation sans remplacement
        for (int i = ranking.Count - 1; i > 0; i--)
        {
            int j = rng.Next(i + 1);
            string tmp = ranking[i];
            ranking[i] = ranking[j];
            ranking[j] = tmp;
        }
        profile.Add(ranking);
    }

    return profile;
}


### Exercice 1 : score de Kemeny sur des profils aleatoires

**Objectif** : a partir de `random_profile` (exemple guide ci-dessus) et de `KemenyBruteForce`, ecrire `KemenyScoresRandom(nVoters, alternatives, nProfiles, seed)` qui tire `nProfiles` profils aleatoires, calcule pour chacun le score de Kemeny optimal (le premier element du triplet rendu par `KemenyBruteForce`) et retourne la liste de ces scores.

**Critere de verification** : chaque score est un entier entre 0 et `n_voters * m(m-1)/2`, et deux appels avec le meme `seed` rendent la meme liste.


In [9]:
// Exercice 1 : score de Kemeny sur des profils aleatoires
static List<int> KemenyScoresRandom(int nVoters, List<string> alternatives, int nProfiles, int? seed = null)
{
    // TODO etudiant : tirer nProfiles profils aleatoires avec RandomProfile,
    // puis renvoyer pour chacun le score de Kemeny optimal (KemenyBruteForce).
    Console.WriteLine("Exercice 1 a completer : renvoyer une liste vide de scores de Kemeny.");
    return new List<int>();
}

Console.WriteLine("Exercice 1 a completer");

Exercice 1 a completer


#### Indication pour l'exercice 1

La generation aleatoire utilise un melange de Fisher-Yates (`System.Random(seed)`) qui produit une permutation aleatoire sans remplacement. Pour `m=4`, l'enumeration Kemeny necessite `4! = 24` permutations, instantane. Avec `seed=42` (par exemple), le profil doit etre reproductible -- utiliser le meme seed pour reproduire les resultats.


### Exercice 2 : Ranked Pairs vs Copeland

**Objectif** : Implementer la regle Copeland (deja presente dans SC-03) et comparer avec Ranked Pairs sur une serie de profils tests. Identifier les cas ou les deux regles donnent le meme vainqueur, et les cas ou elles divergent.


In [10]:
#nullable enable

// Exercice 2 : Ranked Pairs vs Copeland
static List<string>? CopelandRule(List<List<string>> profile)
{
    // TODO etudiant : regle de Copeland (reimplementation succincte).
    // Pour chaque paire (x, y), winner est celui qui l'emporte par majorite.
    // Le score Copeland d'un candidat = (nb victoires) - (nb defaites).
    Console.WriteLine("Exercice 2 a completer");
    return null;
}

static Dictionary<string, object> CompareRpVsCopeland(List<List<string>> profile, List<string> alternatives)
{
    // Compare Ranked Pairs et Copeland sur un profil donne.
    var (rpOrder, rpWinners) = RankedPairsEngine.RankedPairs(profile, alternatives);
    var cpOrder = CopelandRule(profile);
    string? cpWinner = (cpOrder != null && cpOrder.Count > 0) ? cpOrder[0] : null;
    return new Dictionary<string, object>
    {
        ["rp_winner"] = rpWinners,
        ["cp_winner"] = cpWinner ?? "(non calcule)",
        ["agree"] = cpWinner != null && rpWinners.Contains(cpWinner),
    };
}

// Tests : remplacer par des profils ou les regles divergent
// Indice : Copeland ne respecte pas forcement IIA, donc sur des profils
// construits specialement, les deux regles peuvent diverger.
var result1 = CopelandRule(divergentProfile);
Console.WriteLine($"Copeland order (test) : {(result1 == null ? "(non calcule)" : string.Join(" > ", result1))}");

var comparison1 = CompareRpVsCopeland(divergentProfile, alts);
Console.WriteLine($"RP winners : [{string.Join(", ", (List<string>)comparison1["rp_winner"])}]");
Console.WriteLine($"Copeland winner : {comparison1["cp_winner"]}");
Console.WriteLine($"Accord : {comparison1["agree"]}");

Exercice 2 a completer


Copeland order (test) : (non calcule)


Exercice 2 a completer


RP winners : [B]


Copeland winner : (non calcule)


Accord : False


### Exercice 3 : Dodgson winner sur un profil aleatoire

**Objectif** : Comparer Dodgson et Kemeny sur des profils aleatoires de 4 alternatives et identifier les cas ou les vainqueurs coincident.


In [11]:
// Exercice 3 : Dodgson vs Kemeny sur profils aleatoires 4x4
static void Exercise3DodgsonVsKemeny()
{
    // TODO etudiant :
    // 1. Generer 5 profils aleatoires (4 electeurs, 4 alternatives, seed=42)
    // 2. Calculer Kemeny winner (brute-force enumeration, 4! = 24 permutations)
    // 3. Calculer Dodgson winner (programmation dynamique)
    // 4. Comparer les resultats
    Console.WriteLine("Exercice 3 a completer : comparer Dodgson et Kemeny sur 5 profils aleatoires (4 electeurs, 4 alternatives, seed=42).");
}


// Tests : a executer apres implementation
Exercise3DodgsonVsKemeny();

Exercice 3 a completer : comparer Dodgson et Kemeny sur 5 profils aleatoires (4 electeurs, 4 alternatives, seed=42).


#### Indication pour l'exercice 3

Pour `m=4` alternatives, l'enumeration Kemeny necessite `4! = 24` permutations, instantane. Pour Dodgson, la programmation dynamique conserve des vecteurs de victoires dont le nombre augmente avec les electeurs et les alternatives. Les Kemeny winners et Dodgson winners coincident sur certaines classes de profils (ceux ou Kemeny winner est deja Condorcet winner, score Dodgson = 0) mais peuvent diverger sur d'autres (voir Handbook ch. 5 §5.4 pour les exemples theoriques).


***

## References bibliographiques

- Kemeny, J.G. (1959). Mathematics without numbers. Daedalus 88(4):577-591.
- Young, H.P. & Levenglick, A. (1978). A consistent extension of Condorcet's election principle. SIAM Journal on Applied Mathematics 35(2):285-300.
- Tideman, T.N. (1987). Independence of clones as a criterion for voting rules. Social Choice and Welfare 4(3):185-206.
- Dodgson, C. (1876). A method of taking votes upon more than two issues. Manuscrit non publie, reimprime dans McLean, I. & Urken, A.B. (1995), Classics of Social Choice, University of Michigan Press, pp. 273-287.
- Caragiannis, I., Hemaspaandra, E. & Hemaspaandra, L. (2016). Dodgson's Rule and Young's Rule. In *Handbook of Computational Social Choice* (Brandt, Conitzer, Endriss, Lang & Procaccia ed.), ch. 5, pp. 121-143. Cambridge University Press. **Ancrage du carnet pour Dodgson** : §5.3 Winner-Problem Complexity. (L'article *Computational Social Choice Meets Databases* existe, mais il est de Kimelfeld, Kolaitis et Stoyanovich, IJCAI 2018, p. 317-323, et porte sur le vote dans un contexte de bases de donnees -- il n'est pas l'ancre Dodgson.)
- Brill, M. & Fischer, F. (2012). The Price of Neutrality for the Ranked Pairs Method. AAAI. **Ancrage du carnet pour la nuance NP-completude Ranked Pairs** : Theoreme 4.15, Handbook ch. 4 §4.5.
- Brandt, F., Conitzer, V., Endriss, U., Lang, J., Procaccia, A.D. (2016). *Handbook of Computational Social Choice*. Cambridge University Press. ISBN 978-1-107-06043-2. Source biblio cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Social Choice, Voting and Welfare\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`.


> **Lien avec SC-03 et SC-04** : ce carnet complete la ferme des regles de vote presentee dans SC-03 par les regles Kemeny, Ranked Pairs et Dodgson. SC-04 presente l'encodage SAT/Z3 des theoremes d'impossibilite (Arrow, Sen) ; la section 4 de ce carnet transpose cette approche a l'optimisation Kemeny. Les trois carnets (SC-03 polynomial, SC-04 SAT/Z3 theoremes, SC-08 regles computationnelles) forment le triplet pedagogique du cluster.


***

## À retenir

- Kemeny maximise les accords individuels ; ses vainqueurs sont les candidats en tête d’au moins un classement optimal.
- Ranked Pairs verrouille les majorités par marge décroissante sans créer de cycle ; le départage des marges égales peut compter.
- Dodgson minimise les swaps adjacents nécessaires pour obtenir un vainqueur de Condorcet. La programmation dynamique présentée convient aux petits profils.

## Vérifiez votre compréhension

1. Pourquoi faut-il optimiser de nouveau pour chaque candidat dans la méthode Z3 ?
2. Pourquoi Ranked Pairs refuse-t-il un arc qui ferme un cycle ?
3. Pourquoi suffit-il de faire monter le candidat pour calculer son score Dodgson ?

<details>
<summary>Éléments de réponse</summary>

1. Un autre classement optimal peut placer ce candidat en tête, même s’il n’est pas en tête du premier modèle trouvé.
2. Le graphe verrouillé doit rester acyclique pour admettre un classement compatible.
3. Les swaps entre autres candidats ne changent aucun de ses duels et ne peuvent donc améliorer son score.

</details>

## Pour aller plus loin

Comparez les règles sur les profils de SC-03 et examinez les différentes possibilités de départage de Ranked Pairs. Les références ci-dessus donnent les résultats de complexité associés.


***

**Navigation** : [Predecessor: [ChoixSocial-07-Committees-Core-CSharp]] | [Index](../README.md)

> Carnet issu de l'EPIC #26534 (parite C# <-> Python des carnets de la serie SocialChoice).
